In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [60]:
%%writefile src/cpbl.py
from pathlib import Path

import pandas as pd
import requests
from bs4 import BeautifulSoup

ua = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/150.0.0.0 Safari/537.36'
path_way = Path(__file__).resolve().parent.parent / "data" / "raw"
half_codes = {'上半季': '1', '下半季': '2'}
rebas_sections = {'上半季': 'top', '下半季': 'bottom'}


def fetch_cpbl_standings(season_code, refresh=False):
    """中職官網：抓一個 SeasonCode 的戰績表 HTML（0 全年、1 上半季、2 下半季）。"""
    path = path_way / f'cpbl_standing_{season_code}.html'
    if path.exists() and not refresh:
        return path.read_text(encoding="utf-8")
    session = requests.Session()
    page = session.get('https://www.cpbl.com.tw/standings/season',
                       headers={'User-agent': ua}, timeout=20)
    soup = BeautifulSoup(page.text, 'html.parser')
    token = soup.find('input', {'name': '__RequestVerificationToken'})['value']
    res = session.post(
        'https://www.cpbl.com.tw/standings/seasonaction',
        headers={
            'User-agent': ua,
            'x-requested-with': 'XMLHttpRequest',
            'referer': 'https://www.cpbl.com.tw/standings/season',
        },
        data={
            '__RequestVerificationToken': token,
            'ExecAction': '',
            'IndexOfPages': '0',
            'Kindcode': 'A',
            'SeasonCode': season_code,
            'GameEndDate': ''
        },
        timeout=20
    )
    res.raise_for_status()
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(res.text, encoding="utf-8")
    return res.text


def parse_cpbl_standings(html, half):
    """從 HTML 的第一張表拆出球隊和勝、敗、和。官網順序是「勝-和-敗」。"""
    table = BeautifulSoup(html, 'html.parser').select_one('table')
    rows = []
    for tr in table.select('tr'):
        num_td = tr.select('td.num')
        name_id = tr.select_one('a')
        if name_id and len(num_td) >= 14:
            win, tie, lose = num_td[1].text.strip().split('-')
            rows.append({'half': half, 'team': name_id.text.strip(),
                         'W': int(win), 'L': int(lose), 'T': int(tie)})
    return pd.DataFrame(rows)


def cpbl_whole_half_standing(refresh=False):
    """中職官網：上、下半季各抓一次，上下接成一張表。"""
    frames = []
    for half, code in half_codes.items():
        frames.append(parse_cpbl_standings(fetch_cpbl_standings(code, refresh), half))
    return pd.concat(frames, ignore_index=True)


Overwriting src/cpbl.py


In [74]:
%%writefile -a src/cpbl.py

def rebas_cpbl_half_standing(section):
    """野球革命：一個半季的戰績（section：top 上半季、bottom 下半季）。"""
    page = requests.get(
        f'https://www.rebas.tw/api/seasons/CPBL-2026-oB/leaders?type=team&section={section}&pa=undefined',
        headers={'User-agent': ua}, timeout=20)
    page.raise_for_status()
    rows = []
    for i in page.json()['data']:
        rows.append({'team': i['team']['name'], 'W': i['wins'], 'L': i['loses'], 'T': i['draws']})
    return pd.DataFrame(rows)


def rebas_whole_half_standing():
    """Plan B：野球革命的上、下半季，格式跟官網版一樣。"""
    frames = []
    for half, section in rebas_sections.items():
        frames.append(rebas_cpbl_half_standing(section).assign(half=half))
    return pd.concat(frames, ignore_index=True)[['half', 'team', 'W', 'L', 'T']]


def half_standings(refresh=False):
    """先試中職官網，失敗才用野球革命。回傳（表格, 來源）。"""
    try:
        df = cpbl_whole_half_standing(refresh)
        if len(df) == 12:
            return df, '中職官網'
        print(f'官網只解析到 {len(df)} 列，改用野球革命')
    except (requests.RequestException, TypeError) as e:
        print(f'中職官網抓不到（{e}），改用野球革命')
    return rebas_whole_half_standing(), '野球革命（備案）'


def cross_check(half, season):
    """上半季 + 下半季 應該等於全年戰績；回傳對不上的球隊。"""
    total = half.groupby('team')[['W', 'L', 'T']].sum()
    full = season.set_index('team')[['W', 'L', 'T']]
    diff = (total - full).abs().sum(axis=1)
    return diff[diff > 0]


Appending to src/cpbl.py


In [76]:
!git add .
!git commit -m "Day 5: 中職官網半季戰績與交叉比對"

[main 86ed225] Day 5: 中職官網半季戰績與交叉比對
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 7 files changed, 788 insertions(+), 61 deletions(-)
 create mode 100644 D04.ipynb
 create mode 100644 src/cpbl.py
